In [ ]:
!nvidia-smi

Fri Oct  9 05:10:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from numba import cuda

if not cuda.is_available():
    raise RuntimeError(
        "GPU unavailable. Enable a GPU in Colab settings."
    )

device = cuda.get_current_device()

name = device.name
if isinstance(name, bytes):
    name = name.decode("utf-8")

print("Device Name:", name)
print("Compute Capability:", device.compute_capability)
print("Multiprocessor Count:", getattr(
    device, "MULTIPROCESSOR_COUNT", "N/A"
))
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)
print("Max Block Dimensions:",
      device.MAX_BLOCK_DIM_X,
      device.MAX_BLOCK_DIM_Y,
      device.MAX_BLOCK_DIM_Z)
print("Warp Size:", device.WARP_SIZE)

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count: 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024 1024 64
Warp Size: 32


In [ ]:
import numpy as np
import pandas as pd
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)

    for i in range(len(arr)):
        out[i] = arr[i] * 2.0

    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)

    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU measurement
    if N <= 1_000_000:
        t0 = time.perf_counter()
        cpu_out = double_cpu_pure(h_in)
        cpu_ms = (time.perf_counter() - t0) * 1000
    else:
        cpu_ms = np.nan

    threads = 256
    blocks = (N + threads - 1) // threads

    # Warm-up: compile the GPU kernel before timing
    warm = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](warm, warm)
    cuda.synchronize()

    # GPU total time: transfer + kernel + transfer back
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks, threads](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    total_ms = (time.perf_counter() - t0) * 1000

    # GPU kernel-only time
    t0 = time.perf_counter()
    double_gpu_kernel[blocks, threads](d_in, d_out)
    cuda.synchronize()
    kernel_ms = (time.perf_counter() - t0) * 1000

    # Check the result
    assert np.allclose(h_out, h_in * 2)

    return cpu_ms, total_ms, kernel_ms

# Use the seed required by your instructor, if specified.
STUDENT_SEED = 1234

sizes = [
    100, 1_000, 10_000, 100_000,
    1_000_000, 5_000_000, 10_000_000
]

rows = []

for N in sizes:
    cpu_ms, total_ms, kernel_ms = benchmark_run(
        N, STUDENT_SEED
    )

    if np.isnan(cpu_ms):
        winner = "CPU skipped"
    elif total_ms < cpu_ms:
        winner = "GPU"
    else:
        winner = "CPU"

    rows.append({
        "Array Size": N,
        "CPU Time (ms)": cpu_ms,
        "GPU Total (ms)": total_ms,
        "GPU Kernel Only (ms)": kernel_ms,
        "Winner": winner
    })

results_df = pd.DataFrame(rows)
display(results_df)

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


,Array Size,CPU Time (ms),GPU Total (ms),GPU Kernel Only (ms),Winner
0,100,0.082446,1.389659,0.111232,CPU
1,1000,0.277792,0.600641,0.062467,CPU
2,10000,2.974047,0.509623,0.081490,GPU
3,100000,25.673977,0.650390,0.079257,GPU
4,1000000,251.421097,4.959798,0.129571,GPU
5,5000000,NaN,14.004910,0.276016,CPU skipped
6,10000000,NaN,25.408987,0.525142,CPU skipped


In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # Deliberate error: missing bounds check
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

try:
    print("Launching defective kernel...")
    defective_kernel[4, 256](d_arr)
    cuda.synchronize()

    h_result = d_arr.copy_to_host()
    print("Execution finished.")
    print("Elements:", len(h_result))
    print("Last element:", h_result[-1])

except Exception as exc:
    print("Observed error type:", type(exc).__name__)
    print("Observed error message:", str(exc))

Launching defective kernel...


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Execution finished.
Elements: 1000
Last element: 999.0


In [ ]:
@cuda.jit
def inspect_threads_kernel(
    log_block, log_thread, log_global, log_active, N
):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads = 8
blocks = (N + threads - 1) // threads
total_threads = blocks * threads

log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)

inspect_threads_kernel[blocks, threads](
    log_b, log_t, log_g, log_a, N
)
cuda.synchronize()

log_b = log_b.copy_to_host()
log_t = log_t.copy_to_host()
log_g = log_g.copy_to_host()
log_a = log_a.copy_to_host()

print("Global ID | Block ID | Thread ID | Status")
for i in range(32, 40):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(log_g[i], log_b[i], log_t[i], status)

print("Total blocks:", blocks)
print("Total threads:", total_threads)
print("Active threads:", int(np.sum(log_a)))
print("Idle guarded threads:", int(total_threads - np.sum(log_a)))

Global ID | Block ID | Thread ID | Status
32 4 0 ACTIVE
33 4 1 ACTIVE
34 4 2 ACTIVE
35 4 3 ACTIVE
36 4 4 ACTIVE
37 4 5 IDLE (GUARDED)
38 4 6 IDLE (GUARDED)
39 4 7 IDLE (GUARDED)
Total blocks: 5
Total threads: 40
Active threads: 37
Idle guarded threads: 3


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        # Deliberate race condition
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [ ]:
N = 50_000
h_ones = np.ones(N, dtype=np.float32)
d_ones = cuda.to_device(h_ones)

threads = 256
blocks = (N + threads - 1) // threads

naive_results = []

for trial in range(1, 6):
    d_total = cuda.to_device(
        np.array([0.0], dtype=np.float32)
    )

    start = time.perf_counter()

    try:
        naive_sum_kernel[blocks, threads](d_ones, d_total)
        cuda.synchronize()

        value = float(d_total.copy_to_host()[0])
        elapsed_ms = (time.perf_counter() - start) * 1000

        naive_results.append({
            "Trial": trial,
            "Target Sum": 50000.0,
            "Naive Output": value,
            "Deviation": 50000.0 - value,
            "Time (ms)": elapsed_ms
        })

    except Exception as exc:
        print("Trial failed:", trial)
        print(type(exc).__name__, str(exc))
        break

import pandas as pd
display(pd.DataFrame(naive_results))

,Trial,Target Sum,Naive Output,Deviation,Time (ms)
0,1,50000.0,2.0,49998.0,57.961719
1,2,50000.0,2.0,49998.0,0.234637
2,3,50000.0,2.0,49998.0,0.160880
3,4,50000.0,2.0,49998.0,0.175045
4,5,50000.0,2.0,49998.0,0.151854


In [ ]:
# Warm up the atomic kernel
warm_total = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)
atomic_sum_kernel[blocks, threads](d_ones, warm_total)
cuda.synchronize()

# Measure a fresh atomic sum
d_atomic_total = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)

start = time.perf_counter()

atomic_sum_kernel[blocks, threads](
    d_ones, d_atomic_total
)
cuda.synchronize()

atomic_time_ms = (time.perf_counter() - start) * 1000
atomic_value = float(d_atomic_total.copy_to_host()[0])

print("Expected sum:", 50000.0)
print("Atomic sum:", atomic_value)
print("Correct:", atomic_value == 50000.0)
print("Atomic kernel time (ms):", atomic_time_ms)

Expected sum: 50000.0
Atomic sum: 50000.0
Correct: True
Atomic kernel time (ms): 0.5010929999116343


In [ ]:
import numpy as np
import math
from numba import cuda

@cuda.jit
def radial_vignette_2d(
    d_canvas, width, height, center_x, center_y
):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(
            float(center_x**2 + center_y**2)
        )

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()
result_canvas = d_canvas.copy_to_host()

print("Matrix shape:", result_canvas.shape)
print("Blocks in X:", blocks_2d[0])
print("Blocks in Y:", blocks_2d[1])
print("Total blocks:", blocks_2d[0] * blocks_2d[1])
print("Center value:", result_canvas[512, 512])
print("Corner value:", result_canvas[0, 0])

Matrix shape: (1024, 1024)
Blocks in X: 64
Blocks in Y: 64
Total blocks: 4096
Center value: 1.0
Corner value: 0.0


In [ ]:
import hashlib
from numba import cuda

STUDENT_ID = "230103071"
CROSSOVER_N = 10000

def generate_lab_checksum(student_id_str, d_canvas, crossover):
    h = hashlib.sha256()

    h.update(student_id_str.strip().encode("utf-8"))

    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode("utf-8")

    h.update(gpu_name)
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(crossover).encode("utf-8"))

    token = h.hexdigest()[:16].upper()

    print("=" * 60)
    print("OFFICIAL VERIFICATION CHECKSUM TOKEN:", token)
    print("=" * 60)

    return token

checksum_token = generate_lab_checksum(
    STUDENT_ID, d_canvas, CROSSOVER_N
)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 7DBCFFD302CB37A3
